In [4]:
import requests
from bs4 import BeautifulSoup
import csv
from pathlib import Path

In [6]:
SOURCE_URL = "https://realpython.github.io/fake-jobs/"

print(SOURCE_URL)

https://realpython.github.io/fake-jobs/


In [7]:
response = requests.get(SOURCE_URL, timeout=10)

response.raise_for_status()

print("Page downloaded successfully.")
print("Status code:", response.status_code)

Page downloaded successfully.
Status code: 200


In [8]:
soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.get_text(strip=True))

Fake Python


In [9]:
job_cards = soup.find_all("div", class_="card-content")

print("Job cards found:", len(job_cards))

Job cards found: 100


In [10]:
print(job_cards[0].prettify()[:3000])

<div class="card-content">
 <div class="media">
  <div class="media-left">
   <figure class="image is-48x48">
    <img alt="Real Python Logo" src="https://files.realpython.com/media/real-python-logo-thumbnail.7f0db70c2ed2.jpg?__no_cf_polish=1"/>
   </figure>
  </div>
  <div class="media-content">
   <h2 class="title is-5">
    Senior Python Developer
   </h2>
   <h3 class="subtitle is-6 company">
    Payne, Roberts and Davis
   </h3>
  </div>
 </div>
 <div class="content">
  <p class="location">
   Stewartbury, AA
  </p>
  <p class="is-small has-text-grey">
   <time datetime="2021-04-08">
    2021-04-08
   </time>
  </p>
 </div>
 <footer class="card-footer">
  <a class="card-footer-item" href="https://www.realpython.com" target="_blank">
   Learn
  </a>
  <a class="card-footer-item" href="https://realpython.github.io/fake-jobs/jobs/senior-python-developer-0.html" target="_blank">
   Apply
  </a>
 </footer>
</div>



In [11]:
job = job_cards[0]

title = job.find("h2", class_="title").get_text(strip=True)
company = job.find("h3", class_="company").get_text(strip=True)
location = job.find("p", class_="location").get_text(strip=True)
job_url = job.find("a").get("href")

print("Job Title:", title)
print("Company:", company)
print("Location:", location)
print("Job URL:", job_url)

Job Title: Senior Python Developer
Company: Payne, Roberts and Davis
Location: Stewartbury, AA
Job URL: https://www.realpython.com


In [12]:
job_listings = []

for job in job_cards:
    title_element = job.find("h2", class_="title")
    company_element = job.find("h3", class_="company")
    location_element = job.find("p", class_="location")
    link_element = job.find("a")

    job_listings.append({
        "job_title": title_element.get_text(strip=True) if title_element else None,
        "company": company_element.get_text(strip=True) if company_element else None,
        "location": location_element.get_text(strip=True) if location_element else None,
        "job_url": link_element.get("href") if link_element else None
    })

print("Jobs scraped:", len(job_listings))

Jobs scraped: 100


In [13]:
job_listings[:3]

[{'job_title': 'Senior Python Developer',
  'company': 'Payne, Roberts and Davis',
  'location': 'Stewartbury, AA',
  'job_url': 'https://www.realpython.com'},
 {'job_title': 'Energy engineer',
  'company': 'Vasquez-Davidson',
  'location': 'Christopherville, AA',
  'job_url': 'https://www.realpython.com'},
 {'job_title': 'Legal executive',
  'company': 'Jackson, Chambers and Levy',
  'location': 'Port Ericaburgh, AA',
  'job_url': 'https://www.realpython.com'}]

In [14]:
for field in ["job_title", "company", "location", "job_url"]:
    missing = sum(
        1 for job in job_listings
        if not job[field]
    )

    print(f"{field}: {missing} missing")

job_title: 0 missing
company: 0 missing
location: 0 missing
job_url: 0 missing


In [15]:
data_folder = Path("data")
data_folder.mkdir(exist_ok=True)


In [16]:
csv_file = data_folder / "job_listings.csv"

fieldnames = [
    "job_title",
    "company",
    "location",
    "job_url"
]

with open(csv_file, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)

    writer.writeheader()
    writer.writerows(job_listings)

print(f"CSV saved to: {csv_file}")

CSV saved to: data\job_listings.csv


In [17]:
with open(csv_file, "r", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    saved_jobs = list(reader)

print("Records saved:", len(saved_jobs))
print("\nFirst record:")
print(saved_jobs[0])

Records saved: 100

First record:
{'job_title': 'Senior Python Developer', 'company': 'Payne, Roberts and Davis', 'location': 'Stewartbury, AA', 'job_url': 'https://www.realpython.com'}


In [18]:
with open(csv_file, "r", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    data = list(reader)

print(f"Dataset loaded successfully: {len(data)} records")

Dataset loaded successfully: 100 records


In [19]:
print("Columns:")
for column in data[0].keys():
    print("-", column)

Columns:
- job_title
- company
- location
- job_url


In [22]:
## Dataset Overview

##Before performing further analysis, we examine the structure and basic
##quality of the collected dataset.

In [21]:
print("Number of records:", len(data))
print("Number of columns:", len(data[0]))

Number of records: 100
Number of columns: 4


In [23]:
companies = {job["company"] for job in data}

print("Unique companies:", len(companies))

Unique companies: 99


In [24]:
print("Sample companies:")

for company in list(companies)[:10]:
    print("-", company)

Sample companies:
- Bradley LLC
- Stewart-Alexander
- Taylor PLC
- Fisher, Ryan and Coleman
- Williams, Peterson and Rojas
- Waters, Wilson and Hoover
- Smith PLC
- Snyder-Lee
- Spencer and Sons
- Gomez-Carroll


In [25]:
locations = {}

for job in data:
    location = job["location"]

    locations[location] = locations.get(location, 0) + 1

print("Number of unique locations:", len(locations))

Number of unique locations: 100


In [26]:
sorted_locations = sorted(
    locations.items(),
    key=lambda x: x[1],
    reverse=True
)

for location, count in sorted_locations[:10]:
    print(f"{location}: {count}")

Stewartbury, AA: 1
Christopherville, AA: 1
Port Ericaburgh, AA: 1
East Seanview, AP: 1
North Jamieview, AP: 1
Davidville, AP: 1
South Christopher, AE: 1
Port Jonathan, AE: 1
Osbornetown, AE: 1
Scotttown, AP: 1


In [27]:
company_counts = {}

for job in data:
    company = job["company"]

    company_counts[company] = company_counts.get(company, 0) + 1

sorted_companies = sorted(
    company_counts.items(),
    key=lambda x: x[1],
    reverse=True
)

for company, count in sorted_companies[:10]:
    print(f"{company}: {count}")

Garcia PLC: 2
Payne, Roberts and Davis: 1
Vasquez-Davidson: 1
Jackson, Chambers and Levy: 1
Savage-Bradley: 1
Ramirez Inc: 1
Rogers-Yates: 1
Kramer-Klein: 1
Meyers-Johnson: 1
Hughes-Williams: 1


In [29]:
import pandas as pd

ModuleNotFoundError: No module named 'pandas'